# Strategy Research Walkthrough

Build, backtest, and analyze a momentum strategy from scratch using `quant.engine`.

In [ ]:
import pandas as pd
import numpy as np
from engine import Strategy, Signal, Engine, BacktestConfig, DataFrameSource, summary
from engine.report import report_generate

print("Engine ready.")

## 1. Create Synthetic Data

Simulate 2 years of daily data for a small universe.

In [ ]:
np.random.seed(42)
n = 504  # ~2 years of trading days
dates = pd.date_range("2024-01-01", periods=n, freq="D")

close = pd.DataFrame({
    "SPY": 450 + np.cumsum(np.random.randn(n) * 1.5 + 0.03),
    "QQQ": 380 + np.cumsum(np.random.randn(n) * 2.0 + 0.04),
    "IWM": 200 + np.cumsum(np.random.randn(n) * 1.8 + 0.01),
    "XLE": 85 + np.cumsum(np.random.randn(n) * 1.2 + 0.02),
}, index=dates)

data = DataFrameSource(close=close)
print(f"Universe: {data.universe}")
print(f"Bars: {len(data)}")
close.tail()

## 2. Define a Momentum Strategy

Buy top-N momentum stocks, rebalance monthly.

In [ ]:
class MomentumStrategy(Strategy):
    lookback: int = 60        # 60-day momentum
    top_n: int = 2            # Hold top 2 stocks
    rebalance_days: int = 21  # Rebalance every 21 bars (~monthly)

    def on_init(self, ctx):
        self.bars_since_rebalance = 0

    def on_bar(self, ctx, bar):
        self.bars_since_rebalance += 1
        if self.bars_since_rebalance < self.rebalance_days and bar > self.lookback:
            return []

        momentum = (ctx.data.close.iloc[bar] / ctx.data.close.iloc[bar-self.lookback]) - 1
        top = momentum.nlargest(self.top_n).index.tolist()

        signals = []
        for sym in ctx.universe:
            if ctx.portfolio.has_position(sym) and sym not in top:
                signals.append(Signal.close(sym))
        for sym in top:
            if not ctx.portfolio.has_position(sym):
                signals.append(Signal.target(sym, weight=1.0/self.top_n))

        self.bars_since_rebalance = 0
        return signals

print(f"Parameters: {MomentumStrategy().parameters()}")

## 3. Run Backtest

In [ ]:
cfg = BacktestConfig(initial_capital=100_000, slippage_bps=5, commission_bps=1)
engine = Engine(cfg)
result = engine.run(MomentumStrategy(), data)

s = summary(result)
print(f"Strategy: {result.strategy_name}")
print(f"Total Return: {s['total_return']:.2%}")
print(f"Annual Return: {s['annual_return']:.2%}")
print(f"Sharpe Ratio: {s['sharpe_ratio']:.2f}")
print(f"Max Drawdown: {s['max_drawdown']:.2%}")
print(f"Win Rate: {s['win_rate']:.1%}")

## 4. Visualize Equity Curve

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6))

eq = result.portfolio.equity_curve
ax1.plot(eq.index, eq.values, color="#1a1a2e", linewidth=1)
ax1.set_title("Equity Curve")
ax1.set_ylabel("Portfolio Value ($)")
ax1.grid(True, alpha=0.3)

rolling_max = eq.expanding().max()
dd = (eq - rolling_max) / rolling_max
ax2.fill_between(dd.index, dd.values, 0, color="#e74c3c", alpha=0.3)
ax2.plot(dd.index, dd.values, color="#e74c3c", linewidth=0.5)
ax2.set_title("Drawdown")
ax2.set_ylabel("Drawdown %")
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Analyze Performance by Month

In [ ]:
monthly = eq.resample("ME").last().pct_change().dropna()
print(f"Positive months: {(monthly > 0).mean():.1%}")
print(f"Best month: {monthly.max():.2%}")
print(f"Worst month: {monthly.min():.2%}")

monthly.plot(kind="bar", figsize=(12, 3),
             color=["#2ecc71" if v >= 0 else "#e74c3c" for v in monthly])
plt.title("Monthly Returns")
plt.axhline(y=0, color="black", linewidth=0.5)
plt.show()

## 6. Iterate: Try Different Parameters

In [ ]:
results = []
for lookback in [20, 60, 120]:
    for top_n in [1, 2, 3]:
        s = MomentumStrategy()
        s.lookback = lookback
        s.top_n = top_n
        r = Engine(cfg).run(s, data)
        m = summary(r)
        results.append({"lookback": lookback, "top_n": top_n,
                        "sharpe": m["sharpe_ratio"],
                        "return": m["annual_return"],
                        "max_dd": m["max_drawdown"]})

df_results = pd.DataFrame(results).sort_values("sharpe", ascending=False)
df_results.style.background_gradient(subset=["sharpe", "return", "max_dd"])

## 7. Generate Report

In [ ]:
report_generate(result, "momentum_strategy_report.html")
print("Report saved to momentum_strategy_report.html")

## Next Steps

- Replace synthetic data with real data: `df = qd.bars(["SPY","QQQ"], "2024-01-01", "2026-01-01")`
- Add risk rules: `self.add_risk(StopLoss(pct=0.05))`
- Run walk-forward: `WalkForward(strategy, data, cfg).summary()`
- Optimize with `GridSearch(strategy_class, param_grid, data, cfg)`